# A museum heist

(See the [related slide deck](https://docs.google.com/presentation/d/1pgjh_XucMAzxYQCcxn18UnA3iE_kpoiNtiDicviTnZA/edit?usp=sharing))


## Problem setup

Consider a set of $n$ items, $M_n = \{1, 2, \ldots, n\}$. Each item has a value $v_i$ and weight $w_i$. For simplicity we use 1-based indexing in this problem to label items, so $1 \leq i \leq n$.

We are asked to find a subset $\mathcal S\subseteq M_n$ such that for a given value $C_\text{full}$:

\begin{align}
& \sum*{i\in\mathcal S} w_i \leq C_\text{full}\ \text{and} \\
\text{arg}\,\underset{\mathcal S}{\text{max}} & \sum\_{i\in\mathcal S} v_i
\end{align}

In other words, we want the items in the subset $\mathcal S$ to be at or below a weight $C_\text{full}$ and to have the maximum value among all possible subsets that fulfill the weight restriction. We write the total value of the items in the optimal subset as $S(n,C_\text{full})$.

If you are not familiar with the [$\text{arg}\,\underset{\mathcal S}{\text{max}}$ notation](https://en.wikipedia.org/wiki/Arg_max), it means that we are looking for subset $\mathcal S$ whose members add up to the highest value than the members of any other subset.

### Unpacking the $S(n,C_\text{full})$ notation

$S({\color{maroon}n},C_\text{full})$ is the value of all items in the optimal subset of $\{1,2,\ldots,{\color{maroon}n}\}$ whose total weight does not exceed $C_\text{full}$.

\begin{align}
S(n,C_\text{full}) = \text{arg}\,\underset{\mathcal S}{\text{max}} & \sum*{i\in\mathcal S} v_i,\ \text{such that} \\

& \sum*{i\in\mathcal S} w_i \leq C_\text{full}
\end{align}


## Bruce force doesn't work

To apply brute force, we must evaluate each possible subset of $M_n$. For each subset we mut determine, first, if its combined weight is at or below $C_\text{full}$. There are $2^n$ subsets to a set with cardinality $n$. After excluding the heavier subsets, we must find the one with the highest total value ($\sum\limits_{i\in\mathcal S} v_i$). For sufficiently reasonable values, e.g., $n=3000$ which is the number of paintings at the \*Art Institute of Chicago, brute force requires the computation of $2^{3000}$ subsets. That's about $10^{1000}$. Too much work and we will not be done until the Sun is a cold, dark piece of coal.


## Assume solution is known

Assume that the _perfect_ subset $\mathcal S$ is given to us. What would it look like? At the very least, we can tell if the last item in the collection (item labeled $n$) is in $\mathcal S$ or not.

### $n\not\in\mathcal S$

In this case, $\mathcal S$ is also the optimal subset for the immediately smaller set $M_{n-1} = \{1,2,\ldots, n-1\}$. In other words,

$$
S(n, C_\text{full}) = S(n-1, C_\text{full})
$$

One reason that the last item is not in the optimal subset is that it may be too big for us to carry. This happens when $w_n > C_\text{full}$.

### $n\in\mathcal S$

In this case, $\mathcal S$ cannot be an optimal subset of the immediately smaller set $M_{n-1} = \{1,2,\ldots, n-1\}$. That's because $n\in\mathcal S$ and $n\,{\color{red}\not\in} M_{n-1}$.

If, however, we remove $n$ from the optimal subset, then it can be proven that $\mathcal S - \{n\}$ is the optimal subset of $M_{n-1}$.

Removing $n$ from $\mathcal S$ also removes $w_n$ weight from the load. The new load now is $C_\text{full}-w_n$. And perhaps there is a better subset among the first $n-1$ items that fits in this reduced capacity.

So we are left with a dilemma: just because we can take $n$, should we take it? Our decision depends on which option gives us the best value. In looking at the options below it's important to remember that in general, $S(i, r)$ is the sum of values of all items in an optimal subset. This is the optimal subset of $\{1,2,\ldots,i\}$. The total weight of the items in the optimal subset does not exceed $r$.

If **we do not take** $n$, the best value we can hope for is $S(n-1, C_\text{full})$: full load with the best subset among the first $n-1$ items.

If **we do take** $n$, the best value will be $S(n-1, C_\text{full}-w_n)+v_n$: this is the value of a reduced load and the best subset among the first $n-1$ items, _plus_ the value of the last item.


## Partial solutions: finding values of subsets

Putting everything we discussed above, in compact notation, given a set of $\{1,2,\ldots,n\}$ items with weights $w_i$ and values $v_i$, the value $S$ of its optimal subset whose weight does not excedd $C_\text{full}$ is

\begin{align}
S(n,C_\text{full}) =
\begin{cases}
S(n-1,C_\text{full})\ \text{if}\ w_n > C_\text{full},\ \text{otherwise}\\
\max\left( S(n-1,C_\text{full}),\\ S(n-1,C_\text{full}-w_n)+v_n\right)
\end{cases}
\end{align}

This formula can be solved recursively for $S(n-1,C_\text{full})$ and $S(n-1,C_\text{full}-w_n)$. For example

\begin{align}
S(n-1,C_\text{full}) & =
\begin{cases}
S(n-2,C_\text{full})\ \text{if}\ w_{n-1} > C_\text{full},\ \text{otherwise}\\
\max\left( S(n-2,C_\text{full}),\\ S(n-2,C_\text{full}-w_{n-1})+v_{n-1}\right)
\end{cases} \\

S(n-1,C_\text{full}-w_n) & =
\begin{cases}
S(n-2,C_\text{full}-w_n)\ \text{if}\ w_{n-1} > C_\text{full}-w_n,\ \text{otherwise}\\
\max\left( S(n-2,C_\text{full}-w_n),\\ S(n-2,C_\text{full}-w_n-w_{n-1})+v_{n-1}\right)
\end{cases}
\end{align}

This can get pretty messy, pretty fast, so here's the compact notation:
\begin{align}
S(i,r) =
\begin{cases}
S(i-1,r)\ \text{if}\ w_i > r,\ \text{otherwise}\\
\max\left( S(i-1,r),\\ S(i-1,r-w_i)+v_i\right)
\end{cases}
\end{align}

The initial conditions are trivial: $S(i,0) = 0$ is the value of the subset when our carrying capacity is 0 and $S(0,r)$ is the value of the subset from an empty set.

Given these initial conditions, we can now compute $S(i,r)$ for $1\leq i \leq n$ and $1\leq r\leq C_\text{full}$. The computation will produce a $(n+1)\times(C_\text{full}+1)$ matrix, yielding a complexity of about $\mathcal O (nC_\text{full})$ which is far, far, more manageable than brute force's $\mathcal O(2^n)$.


# Implementation of the partial solution

The partial solution produces the matrix with the values $S(i,r)$ for $1\leq i \leq n$ and $1\leq r\leq C_\text{full}$. This requires the initializaton of a matrix to hold these values and a nested loop over the range for $i$ and $r$.


In [56]:
def museum_partial(C_full, w, v):
    # Number of items in the "museum". We are adjusting for 1-based indexing
    # by subtracting 1 from the length of the weights list.
    n = len(w) - 1
    # Initialize the dynamic programming table with zeros.
    # The table has (n+1) rows and (C_full+1) columns.
    # This takes case of initial conditions that require 0 in the
    # first row and first column.
    S = [[0 for _ in range(C_full + 1)] for _ in range(n + 1)]
    # Fill in the dynamic programming table using the recurrence relation.
    for i in range(1, n + 1):
        for r in range(1, C_full + 1):
            # Some verbose variables to help with readability of the recurrence relation.
            available_capacity = r
            weight_of_current_item = w[i]
            capacity_remaining_after_taking_current_item = (
                available_capacity - weight_of_current_item
            )
            best_value_with_current_item = (
                S[i - 1][capacity_remaining_after_taking_current_item] + v[i]
            )
            best_value_without_current_item = S[i - 1][available_capacity]
            if weight_of_current_item <= available_capacity:
                # We can include the current item in the optimal solution because
                # its weight is less than or equal to the available capacity. Just
                # because we can take it, doesn't mean we should. We need to compare
                # the value of taking it versus not taking it. We take the maximum
                # of the two options.
                S[i][r] = max(
                    best_value_without_current_item, best_value_with_current_item
                )
            else:
                # We cannot include the current item in the optimal solution because
                # its weight is greater than the available capacity. Therefore, we
                # can only take the best value without including the current item.
                S[i][r] = best_value_without_current_item
    return S

In [57]:
# Test the forward dynamic programming function with a sample input
if __name__ == "__main__":
    # Weights and values of the items from the companion slide deck. Since we use
    # 1-based indexing, we add a dummy value at index 0.
    w = [None, 3, 2, 1, 4]
    v = [None, 4, 3, 2, 5]
    # The maximum weight capacity of our get-away truck.
    C_full = 6
    # Items in the museum:
    n = len(w) - 1  # Adjust for 1-based indexing
    # Call the forward dynamic programming function.
    result = museum_partial(C_full, w, v)
    print(f"               The weights of the items are: {w[1:]} lbs")
    print(f"                The values of the items are: {v[1:]} million dollars")
    print(f"The maximum weight capacity of the truck is: {C_full} lbs")
    print(f"       The number of items in the museum is: {n}")
    print(
        f"          Max value that can be obtained is: {result[n][C_full]} million dollars"
    )
    print(f"\nDynamic programming table (S):\t{result[0]}")
    for i in range(1, len(result)):
        print(f"\t\t\t\t{result[i]}")

               The weights of the items are: [3, 2, 1, 4] lbs
                The values of the items are: [4, 3, 2, 5] million dollars
The maximum weight capacity of the truck is: 6 lbs
       The number of items in the museum is: 4
          Max value that can be obtained is: 9 million dollars

Dynamic programming table (S):	[0, 0, 0, 0, 0, 0, 0]
				[0, 0, 0, 4, 4, 4, 4]
				[0, 0, 3, 4, 4, 7, 7]
				[0, 2, 3, 5, 6, 7, 9]
				[0, 2, 3, 5, 6, 7, 9]


## The rest of the solution (tracing back $S$)

Table $S$ gives us only the values of optimal subsets for different configurations, ultimately leading to the bottom, right corner. That's where the value of the optimal subset for $M_n = \{1, 2, \ldots, n\}$ can be found.

But that's just a number: how much are the contents of that subset worth. There is no information about the actual contents: what are they?

We can find the contents of the optimal subset by realizing one simple fact: every time we decide to take the $i$-th item, the value of the corresponding optical subset is

$$
S(i,r) = S(i-1, r-w_i)+v_i
$$

Using the $S$ table of the example above, we know that $S(4,6)=9$. This involves the 4th item with weight $w_4 = 4$ and value $v_4=5$. If that 4th item is included in the optimal subset, it means that

\begin{align}
S(4,6) & \overset{{\color{red}?}}= S(4-1, 6-4) + 5 \\
&= S(3,2)+5 \\
& =3+5 = 8
\end{align}

And since we know that $S(4,6)=9$, we conclude that item 4 is _not_ on the optimal subset.

Moving on to the immediately previous item we see that $S(3,6)=9$. This involves a subset of $\{1,2,3\}$ and we want to know if the 3rd item is in that subset. We know that $w_3=1$ and $v_3=2$. Now we can check:

\begin{align}
S(3,6) & \overset{{\color{red}?}} = S(3-1,6-1)+2 \\
& = S(2,5)+2 \\
& = 7+2 = 9
\end{align}

And so item 3 is in the subset. Because we take that item, the capacity is adjusted to $r=6-1$. So when it's time to look if item 2 is in the optimal subset we'll compare the value $S(2,5)$ and $S(1,3)+3$ and so on.

The test above checks if item $i$ is in the optimal subset. We can obtain the same information with an even simpler test.

If item $i$ is in the optimal subset, then

$$S(i,r) \neq S(i-1, r)$$

The code is given below.


In [58]:
def museum_traceback(S, w, C_full):
    # Number of items in the "museum". We are adjusting for 1-based indexing
    # by subtracting 1 from the length of the weights list.
    n = len(w) - 1
    # Initialize an empty list to store the selected items.
    selected_items = []
    # Start from the last item and the full capacity.
    i = n
    r = C_full
    # Trace back through the dynamic programming table to find which items were included.
    while i > 0 and r > 0:
        if S[i][r] != S[i - 1][r]:
            # This means that item i was included in the optimal solution.
            selected_items.append(i)
            # Reduce the remaining capacity by the weight of the included item.
            r -= w[i]
        # Move to the previous item.
        i -= 1
    return selected_items 

In [59]:
def complete_museum_heist(C_full, w, v):
    # First, compute the dynamic programming table.
    S = museum_partial(C_full, w, v)
    # Then, perform traceback to find the selected items.
    selected_items = museum_traceback(S, w, C_full)
    return (
        S[n][C_full],
        selected_items,
    )  # Return the maximum value and the list of selected items

In [60]:
# Demonstration of the complete museum heist function
if __name__ == "__main__":
    # Weights and values of the items from the companion slide deck. Since we use
    # 1-based indexing, we add a dummy value at index 0.
    w = [None, 3, 2, 1, 4]
    v = [None, 4, 3, 2, 5]
    # The maximum weight capacity of our get-away truck. Possible bug:
    # if C_full < smallest weight, we may run into an index error. This
    # will be fixed at later time. For now, chose a value for C_full such that
    # C_full is large enough to accommodate at least one item.
    C_full = 6
    # Items in the museum:
    n = len(w) - 1  # Adjust for 1-based indexing
    # Call the complete museum heist function.
    max_value, selected_items = complete_museum_heist(C_full, w, v)
    print(f"               The weights of the items are: {w[1:]} lbs")
    print(f"                The values of the items are: {v[1:]} million dollars")
    print(f"The maximum weight capacity of the truck is: {C_full} lbs")
    print(f"       The number of items in the museum is: {n}")
    print(f"          Max value that can be obtained is: {max_value} million dollars")
    print(f"          Items selected for the heist are: {selected_items}")

               The weights of the items are: [3, 2, 1, 4] lbs
                The values of the items are: [4, 3, 2, 5] million dollars
The maximum weight capacity of the truck is: 6 lbs
       The number of items in the museum is: 4
          Max value that can be obtained is: 9 million dollars
          Items selected for the heist are: [3, 2, 1]


## How the traceback works?